In [0]:
from pyspark.sql import functions as F

bronze_df = spark.read.table("dbr_dev_ua5816bd.team_tristar_bronze.trips")

latest_date = (
    bronze_df
    .agg(F.max("source_update_date"))
    .first()[0]
)

df = bronze_df.filter(
    F.col("source_update_date") == latest_date
)

df = df.withColumn(
    'route_id',
    F.when(
        F.col('route_id').try_cast('int') <= 0,
        None
    ).otherwise(
        F.col('route_id').try_cast('int')
    )
)

df = df.withColumn(
    'service_id',
    F.when(
        F.trim(F.col('service_id')) == '',
        None
    ).otherwise(
        F.trim(F.col('service_id'))
    )
)

df = df.withColumn(
    'trip_id',
    F.when(
        F.trim(F.col('trip_id')) == '',
        None
    ).otherwise(
        F.trim(F.col('trip_id'))
    )
)

df = df.withColumn(
    'trip_headsign',
    F.when(
        F.trim(F.col('trip_headsign')) == '',
        None
    ).otherwise(
        F.trim(F.col('trip_headsign'))
    )
)

df = df.withColumn(
    'trip_short_name',
    F.when(
        F.trim(F.col('trip_short_name')) == '',
        None
    ).otherwise(
        F.trim(F.col('trip_short_name'))
    )
)

df = df.withColumn(
    'direction_id',
    F.when(
        ~F.col('direction_id').try_cast('int').isin(0, 1),
        None
    ).otherwise(
        F.col('direction_id').try_cast('int')
    )
)

df = df.withColumn(
    'shape_id',
    F.when(
        F.trim(F.col('shape_id')) == '',
        None
    ).otherwise(
        F.trim(F.col('shape_id'))
    )
)

df = df.withColumn(
    'wheelchair_accessible',
    F.when(
        ~F.col('wheelchair_accessible').try_cast('int').isin(0, 1, 2),
        None
    ).otherwise(
        F.col('wheelchair_accessible').try_cast('int')
    )
)

df = df.withColumn(
    'source',
    F.when(
        F.trim(F.col('source')) == '',
        None
    ).otherwise(
        F.trim(F.col('source'))
    )
)

df = df.withColumn(
    'source_update_date',
    F.col('source_update_date').try_cast('date')
)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "dbr_dev_ua5816bd.team_tristar_silver.trips"
)

silver_table.alias("silver").merge(
    df.alias("bronze"),
    "silver.trip_id = bronze.trip_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).whenNotMatchedBySourceDelete(
).execute()